# Text Summarization Model Training

This notebook trains an NLP model for text summarization using a sequence-to-sequence architecture.

In [1]:
# Install required packages
import subprocess
subprocess.run(['pip', 'install', 'transformers', 'datasets', 'torch', 'scikit-learn', 'tf-keras', '-q'])

CompletedProcess(args=['pip', 'install', 'transformers', 'datasets', 'torch', 'scikit-learn', 'tf-keras', '-q'], returncode=0)

## 1. Import Libraries

In [2]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

c:\Users\Adarsh Kumar\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Using device: cpu


## 2. Prepare Dataset

For demonstration, we'll use a sample dataset. In production, replace with your WhatsApp chat data.

In [3]:
# Sample training data - replace with actual WhatsApp chat summaries
training_data = [
    {"source": "Hey! How are you doing today?", "summary": "Greeting and inquiry about well-being."},
    {"source": "The meeting has been rescheduled to 3 PM tomorrow in Conference Room B.", "summary": "Meeting rescheduled to 3 PM tomorrow in Conference Room B."},
    {"source": "Can you please send me the project report by end of day? I need it for the client presentation.", "summary": "Request for project report by end of day for client presentation."},
    {"source": "The server is down and the team is working on it. We expect to have it back up within 2 hours.", "summary": "Server outage - team working on restoration, expected 2 hours."},
    {"source": "Thanks for your help with the presentation yesterday. The client loved it and we've secured the deal!", "summary": "Presentation successful - client secured the deal."},
    {"source": "I'm running late to the airport. My flight is delayed by 3 hours so I should still make it.", "summary": "Flight delayed 3 hours, still arriving as planned."},
    {"source": "The budget review meeting is scheduled for next week Tuesday at 10 AM. Please prepare the Q3 financial reports.", "summary": "Budget review meeting next Tuesday 10 AM - prepare Q3 reports."},
    {"source": "Don't forget to submit your timesheet by Friday evening. The payroll department needs it for processing.", "summary": "Reminder: submit timesheet by Friday evening for payroll."},
]

print(f"Loaded {len(training_data)} training samples")

Loaded 8 training samples


## 3. Create Custom Dataset Class

In [4]:
class SummarizationDataset(Dataset):
    def __init__(self, data, tokenizer, max_source_len=128, max_target_len=64):
        self.data = data
        self.tokenizer = tokenizer
        self.max_source_len = max_source_len
        self.max_target_len = max_target_len
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Tokenize source (input text)
        source_encoding = self.tokenizer(
            item['source'],
            max_length=self.max_source_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Tokenize target (summary)
        target_encoding = self.tokenizer(
            item['summary'],
            max_length=self.max_target_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': source_encoding['input_ids'].squeeze(),
            'attention_mask': source_encoding['attention_mask'].squeeze(),
            'labels': target_encoding['input_ids'].squeeze()
        }

print("Dataset class defined successfully")

Dataset class defined successfully


## 4. Initialize Tokenizer and Model

In [ ]:
# Use pre-trained BERT for fine-tuning
model_name = 'bert-base-uncased'
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = BertModel.from_pretrained(model_name)

# Summarization model (Encoder-Decoder architecture)
class SummarizationModel(nn.Module):
    def __init__(self, encoder, vocab_size):
        super().__init__()
        self.encoder = encoder
        self.decoder = nn.LSTM(vocab_size, 768, batch_first=True)
        self.fc = nn.Linear(768, vocab_size)
    
    def forward(self, input_ids, attention_mask, labels):
        # Encode source text
        encoder_outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        encoder_hidden = encoder_outputs.last_hidden_state
        
        # Decode summary
        decoder_output, _ = self.decoder(encoder_hidden)
        logits = self.fc(decoder_output)
        
        # Calculate loss
        loss = nn.CrossEntropyLoss()(logits.view(-1, logits.size(-1)), labels.view(-1))
        return loss

# Initialize model
vocab_size = tokenizer.vocab_size
model = SummarizationModel(bert_model, vocab_size).to(device)
print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Model initialized with 229,082,682 parameters


## 5. Training Configuration

In [6]:
# Create dataset and dataloader
dataset = SummarizationDataset(training_data, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Training settings
learning_rate = 2e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
num_epochs = 10

print(f"Training configuration:")
print(f"  - Batch size: 2")
print(f"  - Learning rate: {learning_rate}")
print(f"  - Epochs: {num_epochs}")
print(f"  - Training samples: {len(training_data)}")

Training configuration:
  - Batch size: 2
  - Learning rate: 2e-05
  - Epochs: 10
  - Training samples: 8


## 6. Training Loop

In [7]:
model.train()
training_losses = []

for epoch in range(num_epochs):
    total_loss = 0
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Forward pass
        loss = model(input_ids, attention_mask, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(dataloader)
    training_losses.append(avg_loss)
    
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch + 1}/{num_epochs} - Loss: {avg_loss:.4f}")

print("\nTraining completed!")

RuntimeError: input.size(-1) must be equal to input_size. Expected 30522, got 768

## 7. Save Model

In [ ]:
# Save the trained model
model_path = 'd:\\coding\\github\\wa_chat_summariser\\model\\summarizer_model.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'tokenizer': tokenizer,
    'vocab_size': vocab_size,
}, model_path)

print(f"Model saved to: {model_path}")

## 8. Inference Test

In [ ]:
def summarize(text, model, tokenizer, max_length=64):
    model.eval()
    with torch.no_grad():
        inputs = tokenizer(text, return_tensors='pt', max_length=128, truncation=True)
        input_ids = inputs['input_ids'].to(device)
        attention_mask = inputs['attention_mask'].to(device)
        
        # Get model prediction (simplified - in production use beam search)
        encoder_outputs = model.encoder(input_ids=input_ids, attention_mask=attention_mask)
        decoder_output, _ = model.decoder(encoder_outputs.last_hidden_state)
        logits = model.fc(decoder_output)
        
        # Greedy decoding
        predicted_ids = torch.argmax(logits, dim=-1)
        summary = tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
    
    return summary

# Test with sample text
test_text = "The quarterly sales meeting has been moved from Wednesday to Friday at 2 PM in the main conference room."
predicted_summary = summarize(test_text, model, tokenizer)
print(f"Input: {test_text}")
print(f"Summary: {predicted_summary}")

---

## Next Steps

1. **Expand Dataset**: Add more training examples for better generalization
2. **Use Larger Model**: Consider BART or T5 for production use
3. **Fine-tune**: Adjust hyperparameters and train longer
4. **Integrate**: Connect to the WhatsApp chat summarizer extension